# **Perineuronal net morphology (PNN) morphology analysis notebook**

<ins>**Author:**</ins> Shannon Rhoads (Github @shanrhoads)

<ins>**Notebook version:**</ins> v1.1

<ins>**Purpose:**</ins> The purpose of this notebook is to facilitate a pipeline for quantitative analysis of perineuronal net (PNN) morphology from high resolution fluorescence microscopy images. The images this pipeline was optimized for were 3D (XYZ) STED micrographs taken with a Leica STELLARIS 8 FALLCON STED microscope. Neurons from mouse tissue were stained with WFA to detect the PNN component N-acetylgalactosamine.

# <mark> TO-DO:
1. <mark> make sure there are output steps for seg and skel in the individual steps section
2. <mark> add data double checking before analysis - from infer-subc v2
3. <mark> update saving of quant to take in info about the file location from the folders

## **Getting started**
#### *Only do this setup process once per computer or location you will run the analysis.*
1. Clone this repository to the local or remote computer you will use to run the analysis:
    - In your computers terminal, naviate to the location you wish to store the clone of the repository
    - Execute the following code in your computer's terminal:
        > ``` Python
        > git clone https://github.com/shanrhoads/PNN-morpho-quant.git
        > ```
2. Follow the instruction in the [env_create.sh](/env_create.sh) file to install the necessary packages into a new Python environment. 
2. Download [Visual Studio Code](https://code.visualstudio.com/Download), or use Jupyter labs (already install in the conda environemnt) to access the files in this repository.
3. Open this file in VSCode or Jupyter Labs.
    - For VSCode, open the program on your computer and use the built in firstory naviation tools to find and open this file.
    - For Jupyter Labs access, open a terminal on your computer, activate your conda environment, and run "jupyter labs" command to initiate the Jupyter labs interface. From there you can navigate to this file using the built in directory navigation tools.
4. Click on `Select Kernel` in the top right; choose `Python Environment...` and either `PNN-morpho` or `base` from the list of options based on your choice above.

## **Notebook organization:**

This notebook includes the following sections that should be run in order for each dataset:
1. **Imports** - this section imports the necessary Python packages and functions to run the analysis pipeline below (required each run).
2. **Testing Analysis Settings (on select single images)** - This section explains the individual steps included in the final segmentation, skeletonization, and quantification batch process functions. When beginning the analysis for a new, independent dataset, utilize this section to optimize the segmentation and skeletonization parameters before batch processing. It is recommended to test the selected settings on a few images across biological replicates (if possible) and experimental conditions to increase the chances of choosing settings that will be broadly applicable for your data. You're chosen settings will never work perfectly one very image, so you are aiming to find a balance of over and undersegmentation, for example, that will work for most images. After applying the chosen settings to an entire dataset or replicate of data, step 4 below then help you refine any images that the chosen settings did not work for.
3. **Batch Process segmentation and skeletonization (all images in one data folder)** - This section allows you to apply your chosen settings to a set of images from a single folder. The segmentation and skeletonization output files will be saved in a separate specified location. This section is intended to be run separately for data from each biological replicate (contained in one folder). You can run multiple batches sequentially if you'd like to process more than one folder of data.
4. **Quality Check segmentations** - This step is necessary to ensure all of the images have a segmentation that accurately reflects the PNN instensity image. Again, the expectation is not that each image will be perfect, but if any images are very far off from accurate, they can easy skew your data. You will use the code blocks included in Step 2 and editting tools in napari to edit any files that had insufficient segmentations.
5. **Batch Process morphological quantification (all images in one data folder)** - Once the segmentations (and skeletons) have been batch processed and visually inspected for accuracy, the data from one biological replicate can be quantified in this step. The inputs include the raw intensity image used for segmentation/skeletonization and the segmentation/skeletonization files. You can run multiple batches sequentially if you'd like to process more than one folder of data.
6. **Summarize quantitative data per image (quantitative data from multiple folders)** - All of the quantitative data from multiple biological replicates is then summarized per cell. The input is intended to include a list of file paths to all of the quantitative data that will be included during statistical analysis (all biological replicates), though is can also be applied to a single folder of data if only one is listed in the input.

## **Recommended data organization:**

It is recommended to maintain the following file structure:

``` bash
experiment-name-1/                                                                      # included as "experiment" metdata
├── Male/                                                                               # included as "sex" metadata
    ├── Pair#/                                                                          # included as "replicate" metadata
    |   ├── WT/                                                                         # input for steps 3 & 5; included as "genotype" metadata
    |   |   └── cell-num_region_subject-ID_decon_ch02.tif                               # raw input file
    |   ├── cKO/                                                                        # input for steps 3 & 5; included as "genotype" metadata
    |   |   └── cell-num_region_subject-ID_decon_ch02.tif                               # raw input file
    |   ├── processing-data_WT-seg-skel/                                                # result of step 3; input for step 5
    |   |   ├── cell-num_region_subject-ID_decon_ch02-PNN_instance_seg.tif              # instance segmentation of the PNN
    |   |   └── cell-num_region_subject-ID_decon_ch02-PNN_skeleton.tif                  # skeleton of the instance segmentation
    |   ├── processing-date_cKO-seg-skel/                                               # result of step 3; input for step 5
    |   |   ├── cell-num_region_subject-ID_decon_ch02-PNN_instance_seg.tif              # instance segmentation of the PNN
    |   |   └── cell-num_region_subject-ID_decon_ch02-PNN_skeleton.tif                  # skeleton of the instance segmentation
    |   ├── processing-date_WT-quant/                                                   # result of step 5; input for step 6
    |   |   └── cell-num_region_subject-ID_decon_ch02-PNN_quantification.csv            # output quantification (one row of data per PNN piece quantified)
    |   └── processing-date_cKO-quant/                                                  # result of step 5; input for step 6
    |       └── cell-num_region_subject-ID_decon_ch02-PNN_quantification.csv            # output quantification (one row of data per PNN piece quantified)
    └── Pair#/                                                                          # included as "replicate" metadata
|   |   └── ...
├── Female/
|   └── ...
├── Summary-quantification/                                                         # result of step 6
    └── summary-stats.csv                                                           # per image summary statistics (one row of data per image)
experiment-name-2/
└── ...
```

__________
## **STEP 1. Imports:**
Below, the packages/functions necessary for this analysis are imported. This should be run everytime any part of this notebook is to be run.

In [ ]:
# IMPORTS
from pathlib import Path
from typing import Union, List #,Tuple, Any
import time

from bioio import BioImage
# from bioio.writers import OmeTiffWriter
# from tifffile import imwrite

import napari

# from dask_image.imread import imread
import numpy as np
import skimage
import skan
import matplotlib.pyplot as plt
from scipy import stats
# import infer_subc

import sys
sys.path.insert(0, str(Path("..").resolve()))
from src.image_processing import skeletonize_plus, batch_PNN_seg_skel
from src.quantification import surface_area_from_props, batch_PNN_quant, batch_summarize_quant

import pandas as pd
pd.set_option('display.max_columns', None)

----------

## **STEP 2. Testing Analysis Settings (on select single images):**

Below, you will find the individual steps that are used to process the image segmentation, skeletonization, and quantification. This section only needs to be run if you are optimizing settings to use in batch processing segmentation/skeletonization or modifying particular files after batching is finished.

### **1. Read file and metadata**

#### **1A. List files in path**

Specify the following information:
- `file_path`: file path where the input images are located written as a string
- `file_type`: input file type as a string (e.g., ".tif")

Then run the cell below to read in the list of files of the specified type from the specified location. A Napari window will also pop up. The outputs of each processing step below will be added to the window as new layers.

In [ ]:
### USER INPUTS ###
file_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Male/Pair 5/3D STED/cKO"  # OPTIONS:  WT" cKO"
file_type = ".tif"



### PROCESSING - no edits below ###
# open a Napari viewer window to visualize images & processing steps
viewer = napari.Viewer()

# create sorted list of files in directory with specified file type
file_list = sorted(Path(file_path).glob(f"*{file_type}"))

# print list of files with associated index number for selection below
pd.set_option('display.max_colwidth', None)
pd.DataFrame({"Image Name":file_list})

#### **1B. Select file of interest for testing**

Specify the following information:
- `file_index`: the index of the image you would like to look at in the analysis below. The index value is found to the left of the file paths listed in the table above.

Then run the cell below to read  the image
 and view some of its metadata.

In [ ]:
### USER INPUTS ###
file_index = 2  # change this number to select a different file from the list above



### PROCESSING - no edits below ###
# read in file
raw_file = BioImage(str(file_list[file_index]))
raw_img = raw_file.data
metadata = raw_file.standard_metadata

# save relevant metadata information as objects for use later
voxel_size_ZYX = (raw_file.physical_pixel_sizes.Z, raw_file.physical_pixel_sizes.Y, raw_file.physical_pixel_sizes.X)

# print relevant info about the image
print("File shape:", raw_file.shape)
print("Dimensions:", raw_file.dims)
print("Image channels:", raw_file.channel_names)
print("Voxel size:", raw_file.physical_pixel_sizes)



### ALTERNATIVE - if .lif file/metadata desired ###
# # read in .lif file
# lif_path = r"W:\Baldwin Lab\Hayli Spence-Osorio\PNN Project Data\Pair 4\Metadata\250429_HSO_PNN_Pair_4_Day_1.lif"
# lif_imgs = BioImage(lif_path)

# # extract raw image data and metadata
# lif_img_raw = lif_imgs.data
# lif_metadata = lif_imgs.metadata

# # view image in napari
# viewer.add_image(lif_img_raw, scale=voxel_size_ZYX, name="Deconvolved PNN image")

#####################################################################
### DEPRICATED ###
# # alternative read approach using dask array
# # the dask array approach before is formatted to utilize a series of tif images that separate channels, z's, time, etc.. 
# # Zarr formatting may still be the best approach for single files
# stack = imread(str(file_list[file_index]))
# napari.imshow(stack, multiscale=False)

#### **1C. Select small subregion in image to speedup processing below**

Specify the following information:
- `use_small_region`: True/False to indicate if you would like to only process a smaller portion of the image (memory saving step) in the steps below, or not. *Note: if you would like to change which chunk of the image you are selecting, you can adjust the coordinates being selected within the square brackets following np.squeeze()[HERE].* 

Then run the cell below to select the specified small portion of your image, or the entire image. This is the image you will use to optimize the settings for each step below. The output can be visualized in the Napari window.

In [ ]:
### USER INPUTS ###
use_small_region = True  # set to True to use a small region of the image for testing purposes 


### PROCESSING - no edits below ###
# select small region or full image for testing analysis
if use_small_region:
    test_img = np.squeeze(raw_img)[30:45, 800:1250, 600:1050]
else:
    test_img = np.squeeze(raw_img)

# visualize image in napari
viewer.layers.clear()
viewer.add_image(test_img, scale=voxel_size_ZYX, name="Deconvolved PNN image")
print("The test image has been added to the Napari viewer.")

### **2. Segment PNN**

#### **2A. Rescale intensity values**


No user input is required.

The cell below rescales the intensity values per image so the max value is always 1 and the minimum value is always. This should help normalize the segmentation outcomes across images if the value ranges vary by image.

In [ ]:
### PROCESSING - no edits below ###
# calculate min/max values
strech_min = test_img.min()
strech_max = test_img.max()

# rescale image
rescale = (test_img - strech_min + 1e-8) / (strech_max - strech_min + 1e-8)

#### ~~**2A. Background subtraction**~~ <mark> **NOT USED AS IT TAKES A TON OF TIME TO PROCESS** - will try without this first </mark>

~~[`Rolling ball background subtraction`](https://scikit-image.org/docs/0.25.x/auto_examples/segmentation/plot_rolling_ball.html) can be used to remove non-uniform background from images before segmentaiton or intensity measurements. In this process the amount of background per pixel/voxel is calculated from a region about the image (here defined as the radius). The radius (in voxels) can be adjusted to modify the effects of the rolling ball algorithm; it should be larger than the largest object/structure of interest in your image.~~



In [ ]:
# ### USER INPUT ###
# bg_radius = 50

# # calculate background per pixel using the rolling ball method
# bg = skimage.restoration.rolling_ball(test_img, radius=bg_radius)
# bg_subtract = test_img - bg

# # visualize output
# viewer.add_image(bg_subtract, scale=voxel_size_ZYX, name="Background subtracted")

#### ~~**2A. Denoising**~~ <mark> **NOT USED AS IT TAKES A TON OF TIME TO PROCESS** - will try without this first </mark>

~~There is quite a bit of speckley noise in your image that is making segmentation of the PNN intensity more difficult. Below, [`skimage.restoration`](https://scikit-image.org/docs/0.25.x/api/skimage.restoration.html#skimage.restoration.denoise_bilateral) module is used to denoise the image.~~

In [ ]:
# denoised = skimage.restoration.denoise_nl_means(test_img, patch_size=50, patch_distance=100)

# viewer.add_image(denoised, scale=voxel_size_ZYX, name="Denoised")

#### **2B. Smoothing**

[`Gaussian`](https://scikit-image.org/docs/0.25.x/api/skimage.filters.html#skimage.filters.gaussian) and [`median`](https://scikit-image.org/docs/0.25.x/api/skimage.filters.rank.html#skimage.filters.rank.median) smoothing filters are commonly used to smooth images and reduce certain types of noise, liek the high salt and pepper noise I think is present in your WFA stained images. The filters smooth the image different ways and are commonly used in combination. I've included both options here with sigma/size filter values that can be used to adjust how much smoothing occurs (large values = more smoothing).

Specify the following:
- `gaus_sigma`: the sigma value used for gaussian smoothing. The higher the number, the more the image is smoothed
- `med_size`: the size value used for the median smoothing filter. The higher the number, the more the image is smoothed

Then run the cell below to apply the smoothing filters. The result can be visualized in the Napari window.

In [ ]:
### USER INPUT ###
gaus_sigma = 2
med_size = 8



### PROCESSING - no edits below ###
# applying smoothing filters
if gaus_sigma:
    smoothed = skimage.filters.gaussian(rescale, sigma=gaus_sigma)
else:
    smoothed = rescale

if med_size:
    fp = skimage.morphology.footprint_rectangle((round(med_size*(voxel_size_ZYX[0]/float(np.max(voxel_size_ZYX)))), 
                                                 round(med_size*(voxel_size_ZYX[1]/float(np.max(voxel_size_ZYX)))), 
                                                 round(med_size*(voxel_size_ZYX[2]/float(np.max(voxel_size_ZYX))))))
    smoothed = skimage.filters.median(smoothed, footprint=fp)

# visualize outputs
viewer.add_image(smoothed, scale=voxel_size_ZYX, name=f"Smoothed: gaus={gaus_sigma}, med={med_size}")

#### **2C. Thresholding** (multiple options below; choose 1)

There any many types of thresholding methods available in Python. The simplest form is to apply a manual threshold cutoff value to the image (all pixels/voxels with this intensity value and above will be included in the segmentation). Alternatively, the [`skimage`](https://scikit-image.org/docs/stable/api/skimage.filters.html) package has many mathematical approaches to calculate the approate threshold value based on the intensity values in the image within its `threshold` module. These automated threshold can help to adjust segmentation outcomes based on image-to-image variations. 

Both manual and automated thresholding approaches can be applied to the entire image (globally). Automated thresholds can also be applied in a local or adaptive fashion where a small local region surround each voxel area used to set a specific threshold value per voxel. Local thresholding can help to adjust the segmentation outcomes within an image if there are intensity or background variations in different regions.

<ins>**Approach 1:**</ins> Manual, global thresholding

In this approach, a cutoff value, representing the minimum intensity value to include as part of the segmentation, is specified by the user. Any voxels with an intensity value less than the cutoff will be excluded from the semantic segmentation.

Specify the following:
- `manual_cutoff`: The minimum intensity value you wish to include in your segmentation

Then run the cell below to apply the cutoff value to your entire image. The resulting segmentation is output into the Napari window.

In [ ]:
### USER INPUT ###
manual_cutoff = 0.045


### PROCESSING - no edits below ###
# select everything above threshold value for segmentation
seg = smoothed >= manual_cutoff

# visualize segmentation
viewer.add_image(seg, scale=voxel_size_ZYX, name=f"Segmentation: thresh={manual_cutoff}", blending="additive", opacity=0.4, colormap='magenta')

<ins>**Approach 2:**</ins> Automated thresholding

In this approach, the threshold cutoff value per image is calculated based on the range of intensity values within that image. Common methods from the [`skimage.filters`](https://scikit-image.org/docs/stable/api/skimage.filters.html) module can be used.

Specify the following:
- `threshold_method`: automated thresholding method. Options: `'otsu'`, `'li'`, `'yen'`, `'isodata'`, `'triangle'`, `'minimum'`, `'mean'`, `'multiotsu'`
- `adjust`: multiplicative threshold adjustment (`1` = no change, `<1` = include more voxels, `>1` = include fewer voxels)
- `multiotsu_middle_to`: only used when `threshold_method='multiotsu'`; choose `'foreground'` or `'background'` to control whether the middle class is grouped with higher or lower intensities

Then run the cell below to apply your chosen thresholding approach to the entire image. The resulting segmentation is shown in the Napari window.

In [ ]:
### USER INPUT ###
threshold_method = 'multiotsu'      # OPTIONS: 'otsu', 'li', 'yen', 'isodata', 'triangle', 'minimum', 'mean', 'multiotsu'
adjust = 0.55                      # OPTIONS: 1 = no threshold adjustment, <1 = more stuff selected, >1 = less stuff selected
multiotsu_middle_to = 'background'  # OPTIONS: 'foreground', 'background'


### PROCESSING - no edits below ###
# Apply automated threshold based on selected method
if threshold_method == 'otsu':
    thresh_value = skimage.filters.threshold_otsu(smoothed)
elif threshold_method == 'multiotsu':
    thresholds = skimage.filters.threshold_multiotsu(smoothed, classes=3)
    if multiotsu_middle_to == 'foreground':
        thresh_value = thresholds[0]  # select the second highest threshold
    elif multiotsu_middle_to == 'background':
        thresh_value = thresholds[1]   # select the lowest threshold
    else:
        raise ValueError(f"Unrecognized multiotsu middle to option: {multiotsu_middle_to}")
elif threshold_method == 'li':
    thresh_value = skimage.filters.threshold_li(smoothed)
elif threshold_method == 'yen':
    thresh_value = skimage.filters.threshold_yen(smoothed)
elif threshold_method == 'isodata':
    thresh_value = skimage.filters.threshold_isodata(smoothed)
elif threshold_method == 'triangle':
    thresh_value = skimage.filters.threshold_triangle(smoothed)
elif threshold_method == 'minimum':
    thresh_value = skimage.filters.threshold_minimum(smoothed)
elif threshold_method == 'mean':
    thresh_value = skimage.filters.threshold_mean(smoothed)
else:
    raise ValueError(f"Unrecognized threshold method: {threshold_method}")

# Apply threshold with optional adjustment
seg_auto = smoothed >= thresh_value*adjust

# Print the calculated threshold value for reference
print(f"Calculated threshold value using {threshold_method}: {thresh_value*adjust}")

# Visualize segmentation
viewer.add_image(seg_auto, scale=voxel_size_ZYX, name=f"Auto seg: {threshold_method} (thresh={thresh_value*adjust})", blending="additive", opacity=0.4, colormap='green')


# ### FOR TESTING ALL THE METHODS ###
# ### Compare multiple automated threshold methods ###
# # the following script can be used to visualize the different thresholding methods available in skimage
# methods = ['otsu', 'li', 'yen', 'isodata', 'triangle', 'minimum', 'mean', 'multiotsu']
# multiotsu_middle = 'foreground'  # OPTIONS: 'foreground', 'background'

# for method in methods:
#     if method == 'otsu':
#             thresh_val = skimage.filters.threshold_otsu(smoothed)
#     elif method == 'multiotsu':
#         thresholds = skimage.filters.threshold_multiotsu(smoothed, classes=3)
#         if multiotsu_middle == 'foreground':
#             thresh_val = thresholds[0]  # select the second highest threshold
#         elif multiotsu_middle == 'background':
#             thresh_val = thresholds[1]   # select the lowest threshold
#         else:
#             raise ValueError(f"Unrecognized multiotsu middle to option: {multiotsu_middle}")
#     elif method == 'li':
#         thresh_val = skimage.filters.threshold_li(smoothed)
#     elif method == 'yen':
#         thresh_val = skimage.filters.threshold_yen(smoothed)
#     elif method == 'isodata':
#         thresh_val = skimage.filters.threshold_isodata(smoothed)
#     elif method == 'triangle':
#         thresh_val = skimage.filters.threshold_triangle(smoothed)
#     elif method == 'minimum':
#         thresh_val = skimage.filters.threshold_minimum(smoothed)
#     elif method == 'mean':
#         thresh_val = skimage.filters.threshold_mean(smoothed)
#     else:
#         raise ValueError(f"Unrecognized threshold method: {method}")
    
#     seg_temp = smoothed >= thresh_val*adjust
#     print(f"{method}: threshold = {thresh_val*adjust}")
#     viewer.add_image(seg_temp, scale=voxel_size_ZYX, name=f"{method} ({thresh_val*adjust})", blending="additive", opacity=0.4, colormap='green')

<ins>**Approach 3:**</ins> Local, automated thresholding

Above, thresholding was applied globally. Here, thresholding is applied locally by estimating a threshold per voxel from nearby intensity values. This can improve segmentation when intensity or background varies across regions of the same image.

Specify the following:
- `local_method`: local thresholding method. Options: `'gaussian'`, `'mean'`, `'median'`, `'otsu'`, `'li'`
- `local_size`: neighborhood size used for local thresholding (must be odd; larger values produce broader local context and are slower)
- `local_adjust`: multiplicative threshold adjustment (`1` = no change, `<1` = include more voxels, `>1` = include fewer voxels)
- `guassian_sigma_local`: gaussian sigma parameter used only when `local_method='gaussian'`

Then run the cell below to generate local-threshold segmentation and display it in Napari.

*Note: this approach is much slower because threshold values are computed locally throughout the image.*

In [ ]:
### USER INPUT ###
# determine the method to use for local thresholding
local_method = 'otsu'       # OPTIONS: 'gaussian', 'mean', 'median', 'otsu', 'li'
local_size = 71            # must be odd; large sizes (takes MORE time) for larger structures, small sizes for smaller structures
local_adjust = 1           # OPTIONS: 1 = no threshold adjustment, <1 = more stuff selected, >1 = less stuff selected

# Some methodsrelevant parameters
guassian_sigma_local = 40    # only used if local_method is 'gaussian'

# rescale and convert to 8-bit for local thresholding
max_val = smoothed.max()
smoothed_8bit = np.round((smoothed / max_val)*255).astype(np.uint8)
smoothed_8bit_downscaled = (smoothed_8bit // 2).astype(np.uint8)



### PROCESSING - no edits below ###
# Calculate global automated threshold value
if local_method == 'otsu':
    # create footprint for local region
    footprint = skimage.morphology.ball(local_size)
    local_otsu_threshold = skimage.filters.rank.otsu(smoothed_8bit_downscaled, footprint)
    seg_local_auto = smoothed_8bit_downscaled >= local_otsu_threshold*local_adjust
elif local_method == 'mean':
    local_mean_threshold = skimage.filters.threshold_local(smoothed_8bit, block_size=local_size, method='mean')
    seg_local_auto = smoothed_8bit >= local_mean_threshold*local_adjust
### MEDIAN LOCAL THRESHOLD NOT WORKING ###
# elif local_method == 'median':
#     local_median_threshold = skimage.filters.threshold_local(smoothed, block_size=local_size, offset=0.1, method='median')
#     seg_local_auto = smoothed >= local_median_threshold
elif local_method == 'gaussian':
    local_gaussian_threshold = skimage.filters.threshold_local(smoothed_8bit, block_size=local_size, method='gaussian', param=guassian_sigma_local)
    seg_local_auto = smoothed_8bit >= local_gaussian_threshold*local_adjust
### NOT TESTED YET ###
# elif local_method in ['li', 'yen']:
#     if local_method == 'li':
#         def thresh_method(neighborhood):
#             funct = skimage.filters.threshold_li(neighborhood)
#             return funct
#     elif local_method == 'yen':
#         def thresh_method(neighborhood):
#             funct = skimage.filters.threshold_yen(neighborhood)
#             return funct
#     elif local_method == 'isodata':
#         def thresh_method(neighborhood):
#             funct = skimage.filters.threshold_isodata(neighborhood)
#             return funct
#     elif local_method == 'triangle':
#         def thresh_method(neighborhood):
#             funct = skimage.filters.threshold_triangle(neighborhood)
#             return funct
#     elif local_method == 'minimum':
#         def thresh_method(neighborhood):
#             funct = skimage.filters.threshold_minimum(neighborhood)
#             return funct
    
#     local_thresholds = skimage.filters.threshold_local(smoothed, block_size=local_size, method='generic', param=thresh_method)
#     seg_local_auto = smoothed >= local_thresholds
else:
    raise ValueError(f"Unrecognized local threshold method: {local_method}")

viewer.add_image(seg_local_auto, scale=voxel_size_ZYX, name=f"Local {local_method.capitalize()} segmentation", blending="additive", opacity=0.4, colormap='cyan')

In [ ]:
### FOR REFERENCE - this works to process local otsu ###
# local_size = 100      ## For [15,450,450] sized image: size=100 -> ~9mins (pretty good outcome); size=11 much faster (created too many smaller objs); size=50 --> 2 mins, 3 sec (similar outcome as manual seg)

# smoothed_uint8 = (smoothed / smoothed.max() * 255).astype(np.uint8)

# footprint = skimage.morphology.ball(local_size)

# local_otsu_threshold = skimage.filters.rank.otsu(smoothed_uint8, footprint)
# binary_local_otsu = smoothed_uint8 > local_otsu_threshold

# viewer.add_image(smoothed_uint8, scale=voxel_size_ZYX, name=f"uint8 smoothed image")
# viewer.add_image(local_otsu_threshold, scale=voxel_size_ZYX, name=f"Local Otsu threshold (radius={local_size})")
# viewer.add_image(binary_local_otsu, scale=voxel_size_ZYX, name=f"Local Otsu segmentation (radius={local_size})", blending="additive", opacity=0.4, colormap='cyan')

#### **2D. Clean-up segmentation**

Segmentations can have errors due to imperfections in the thresholding output. Two different refining setups have been added below: removing small objects and filling small holes.

#### Remove small objects:

Specify the following:
- `obj_min_diameter`: minimum object diameter to keep (in voxels; converted to area/volume threshold in processing)
- `obj_method`: filtering mode, either `'slices'` (2D per Z-slice) or `'3D'` (full-volume filtering)
- `seg`: segmentation mask to refine (`seg`, `seg_auto`, or `seg_local_auto`)

Then run the cell below to remove objects smaller than the selected size threshold.

In [ ]:
### USER INPUTS ###
obj_min_diameter = 10
obj_method = '3D' # OPTIONS: 'slices' or '3D'
seg = seg_local_auto  # choose which segmentation to use for object filtering; OPTIONS: seg, seg_auto, seg_local_auto


### PROCESSING - no edits below ###
# filter objects based on size
if obj_method == 'slices':
    filtered = np.zeros_like(seg)
    for z in range(seg.shape[0]):
        input = seg[z,:,:]
        seg_size_filter = skimage.morphology.remove_small_objects(input, min_size=obj_min_diameter**2)
        input = np.expand_dims(seg_size_filter, axis=0)
        filtered[z,:,:] = input
elif obj_method == '3D':
    filtered = skimage.morphology.remove_small_objects(seg, min_size=obj_min_diameter**3)
else:
    SyntaxError("Unrecognized method chosen. Options include: 'slices' or '3D'.")

# visualize
viewer.add_image(filtered, scale=voxel_size_ZYX, name=f"Filter obj: method={obj_method}, obj={obj_min_diameter}", blending="additive", opacity=0.4, colormap='cyan')

#### Fill small holes

Specify the following:
- `small_hole_diameter_max`: maximum hole diameter to fill (set `0` to disable filling)
- `hole_method`: filling mode, either `'slices'` (2D per Z-slice) or `'3D'` (full-volume filling)

Then run the cell below to fill holes in the filtered segmentation mask.

In [ ]:
### USER INPUTS ###
small_hole_diameter_max = 0
hole_method = 'slices' # OPTIONS: 'slices' or '3D'


### PROCESSING - no edits below ###
# fill holes based on size
if hole_method == 'slices':
    filled = np.zeros_like(filtered)
    for z in range(filtered.shape[0]):
        input = filtered[z,:,:]
        seg_fill_holes = skimage.morphology.remove_small_holes(input, small_hole_diameter_max**2, connectivity=8)
        input = np.expand_dims(seg_fill_holes, axis=0)
        filled[z,:,:] = input
elif hole_method == '3D':
    filled = skimage.morphology.remove_small_holes(filtered, small_hole_diameter_max**3, connectivity=26)
else:
    SyntaxError("Unrecognized method chosen. Options include: 'slices' or '3D'.")

# visualize
viewer.add_image(filled, scale=voxel_size_ZYX, name=f"Fill holes: method={hole_method}, hole={small_hole_diameter_max}", blending="additive", opacity=0.4, colormap='cyan')

#### **2E. Create instance segmentation**

The instance segmentation will not be used for skeletonization, but will be able to tell us how many separate pieces of the PNN exist.

In [ ]:
### PROCESSING - no edits below ###
# create instance seg
instance_seg = skimage.morphology.label(filled)

# visualize output
viewer.add_labels(instance_seg, scale=voxel_size_ZYX, name=f"Instance segmentation", opacity=0.4)

### **3. Skeletonize segmentation**

[Skeletonization](https://scikit-image.org/docs/0.25.x/auto_examples/edges/plot_skeleton.html) is the process by which a 2D or 3D object is narrowed to a pixel-wide representation of the original area/volume. Then, the skeleton is converted into a network graph using the [`skan`](https://skeleton-analysis.org/stable/) package for easier downstream manipulations.

The napari visualization in the next step shows the skeleton branches color coded by length.

#### **3A. Create skeleton & summarize information about each individual branch**

The summary table includes the following information:
- `skeleton_id`: Unique ID for each separate skeleton object (derived from the instance segmentation label ID)
- `branch_id`: Sequential unique ID for each branch/path in the skeleton (0 to n_paths-1)
- `random_branch_id`: Randomly permuted branch ID for visualization purposes
- `node_id_src`: Pixel index ID of the source/starting node of the branch
- `node_id_dst`: Pixel index ID of the destination/ending node of the branch
- `branch_distance`: Total distance along the branch path in physical units (µm), accounting for pixel spacing
- `branch_type`: Classification of the branch topology:
    - 0 = endpoint-to-endpoint (isolated branch)
    - 1 = junction-to-endpoint
    - 2 = junction-to-junction
    - 3 = isolated cycle
- `mean_pixel_value`: Mean intensity value of pixels along the branch path
- `stdev_pixel_value`: Standard deviation of intensity values along the branch path
- `image_coord_src_0`, `image_coord_src_1`, `image_coord_src_2`: Source node coordinates in image space (pixels) for Z, Y, X respectively
- `image_coord_dst_0`, `image_coord_dst_1`, `image_coord_dst_2`: Destination node coordinates in image space (pixels) for Z, Y, X respectively
- `coord_src_0`, `coord_src_1`, `coord_src_2`: Source node coordinates in physical space (µm) for Z, Y, X respectively
- `coord_dst_0`, `coord_dst_1`, `coord_dst_2`: Destination node coordinates in physical space (µm) for Z, Y, X respectively
- `euclidean_distance`: Straight-line (Euclidean) distance between source and destination nodes in physical units (µm)

In [ ]:
### PROCESSING - no edits below ###
# create skeleton from the instance segmentation
labeled_skel, skeleton = skeletonize_plus(instance_seg)

# convert to network graph based  labeled skeleton object
skel_g = skan.Skeleton(labeled_skel, spacing=voxel_size_ZYX, value_is_height=False)

# visualize output
viewer.layers.clear()
viewer.add_image(smoothed, scale=voxel_size_ZYX, name=f"Smoothed Input")
viewer.add_image(filled, scale=voxel_size_ZYX, name=f"Segmentation", blending="additive", opacity=0.3)
viewer.add_labels(instance_seg, scale=voxel_size_ZYX, name=f"Instance segmentation", blending="additive", opacity=0.8)

all_paths = [skel_g.path_coordinates(i) for i in range(skel_g.n_paths)]
paths_table = skan.summarize(skel_g, separator='_')
paths_table.insert(1, 'branch_id', np.arange(skel_g.n_paths))
paths_table.insert(2, 'random_branch_id', np.random.default_rng().permutation(skel_g.n_paths))

# replace skeleton_id with the ID of the organelle object from which the skeleton branch originated
if not np.any(skel_g.path_stdev()):
    # checker to see if all path points and nodes come from the same object
    paths_table['skeleton_id'] = skel_g.path_means().astype(int)
else:
    raise ValueError("at least one branch came from different organelle objects")

# calculate the degree of connectivity for each branch end point
endpoints_src = skel_g.paths.indices[skel_g.paths.indptr[:-1]]
endpoints_dst = skel_g.paths.indices[skel_g.paths.indptr[1:] - 1]

deg_src = skel_g.degrees[endpoints_src]
deg_dst = skel_g.degrees[endpoints_dst]
paths_table['deg_src'] = deg_src
paths_table['deg_dst'] = deg_dst

# view skeleton and paths table
viewer.add_shapes(all_paths, shape_type='path', properties=paths_table, edge_width=1, edge_color='skeleton_id', edge_colormap='hsv', scale=voxel_size_ZYX, opacity=1, name="Skeleton")
paths_table.set_index(['skeleton_id', 'branch_id']).sort_index()

#### **3B. Refine skeleton**
Based on measures calculated by the `skan` package, refine the skeleton by removing short endpoint branches that are likely noise.

Specify the following:
- `min_branch_len`: minimum endpoint branch length (in microns) to retain during skeleton pruning

Use the histogram and Napari overlay to choose a branch-length cutoff that preserves meaningful structure, then run the pruning cell below.

In [ ]:
### PROCESSING - no edits below ###
# plot the histogram of branch lengths using plt.hist from matplotlib package
b_len = paths_table['branch_distance']
possible_range = (b_len.min(), b_len.max())
num_bins = round(possible_range[1]-possible_range[0])

plt.hist(paths_table['branch_distance'], bins=num_bins*50, range=possible_range, density=False, alpha=0.7)
plt.title('Branch Length Histogram')
plt.xlabel('Branch Length (µm)')
plt.ylabel('Number of Branches')
plt.grid(axis='y', alpha=0.75)
plt.show()

Using the histogram above and napari visualization, choose the minimum branch length you want to keep within your skeleton object

In [ ]:
### USER INPUT ###
min_branch_len = 0.3


### PROCESSING - no edits below ###
# select only branches that have end points
endpoint_indices = paths_table.index[(paths_table['deg_src'] == 1) | (paths_table['deg_dst'] == 1)]
print(f"Number of endpoint branches: {len(endpoint_indices)}")

# select branches that are the min size or below
short_paths_indices = paths_table.index[paths_table['branch_distance'] < min_branch_len]
print(f"Number of branches shorter than {min_branch_len} µm: {len(short_paths_indices)}")

# find indices that are endpoints and shorter than the determined size
indices_removed = pd.Index(set(endpoint_indices) & set(short_paths_indices))
print(f"Number of branches to be removed (endpoints and short): {len(indices_removed)}")

# remove those branches from the skeleton object & the paths table
pruned_skeleton = skel_g.prune_paths(indices_removed)
paths_table_pruned = paths_table.drop(indices_removed)

# visualize output
all_paths_pruned = [pruned_skeleton.path_coordinates(i) for i in range(pruned_skeleton.n_paths)]
viewer.add_shapes(all_paths_pruned, shape_type='path', properties=paths_table_pruned, edge_width=1, edge_color='skeleton_id', edge_colormap='hsv', scale=voxel_size_ZYX, opacity=1, name="Skeleton pruned")
paths_table_pruned.set_index(['skeleton_id', 'branch_id'], inplace=True)
paths_table_pruned.sort_index(inplace=True)
paths_table_pruned

##### **Test export of skeleton object as image (for batch processing below)**

Run the next cell to confirm the pruned skeleton can be converted back to a labeled image for saving and batch workflows.

In [ ]:
### PROCESSING - no edits below ###
#  convert skeleton to image for visualization
skel_g_image = pruned_skeleton.skeleton_image
print("Skeleton image shape/unique obj IDs:", skel_g_image.shape, np.unique(skel_g_image))

# visualize output (this should be savable as a tiff if desired)
viewer.add_labels(skel_g_image.astype(int), scale=voxel_size_ZYX, name="Pruned skeleton labels", opacity=0.6)

### **4. Measure PNN features**

#### **4A. Object size/shape and intensity measures per PNN objects and whole PNN**

Quantify the instance segmentation per PNN object and for the whole PNN (all segmented objects combined into one).

Specify the following:
- `include_surface_area_metrics`: set to `True` to include surface area and surface-area-to-volume metrics (slower), or `False` to skip those metrics

Then run the cell below to compute regionprops-based morphology and intensity measurements.

In [ ]:
### USER INPUT ###
include_surface_area_metrics = True  # set to True to include surface area calculations in the regionprops analysis




### PROCESSING - no edits below ###

### CONTNUING to quantification
# list properties to include in regionprops analysis
properties = ['label', 'bbox', 'centroid', 'num_pixels', 'area', 'equivalent_diameter', 
              'major_axis_length', 'minor_axis_length', 'extent', 'solidity', 'euler_number',
              'min_intensity', 'max_intensity', 'mean_intensity', 'intensity_std']

# process regionprops analysis for EACH PNN OBJECT SEPARATELY
props_obj = skimage.measure.regionprops_table(label_image=instance_seg, 
                                            intensity_image=test_img,
                                            properties=properties,
                                            spacing=voxel_size_ZYX)

props_obj_tab = pd.DataFrame(props_obj)
props_obj_tab.insert(0, 'object', 'PNN fragment')

if include_surface_area_metrics:
    surface_area_tab = pd.DataFrame(surface_area_from_props(instance_seg, props_obj, voxel_size_ZYX), columns=['surface_area'])
    props_obj_tab.insert(13, 'surface_area', surface_area_tab['surface_area'])

# process regionprops analysis for WHOLE PNN OBJECT (one per image)
# ensure semantic segmentation only constists of 1 object (ID=1)
whole_PNN = (filled>0).astype(np.uint8)

props_whole = skimage.measure.regionprops_table(label_image=whole_PNN, 
                                                intensity_image=test_img,
                                                properties=properties,
                                                spacing=voxel_size_ZYX)
props_whole_tab = pd.DataFrame(props_whole)
props_whole_tab.insert(0, 'object', 'whole PNN')

if include_surface_area_metrics:
    surface_area_tab_whole = pd.DataFrame(surface_area_from_props(whole_PNN, props_whole, voxel_size_ZYX), columns=['surface_area'])
    props_whole_tab.insert(13, 'surface_area', surface_area_tab_whole['surface_area'])


# combine both tables
combined_props_tab = pd.concat([props_obj_tab, props_whole_tab], ignore_index=True)

# rename & add columns for clarity and additional information
combined_props_tab.insert(0, 'image_name', file_list[file_index].name)
combined_props_tab.rename(columns={'area':'volume'}, inplace=True)
combined_props_tab['intensity_sum'] = combined_props_tab['mean_intensity'] * combined_props_tab['num_pixels']
if include_surface_area_metrics:
    combined_props_tab.insert(15, "SA_to_volume_ratio", combined_props_tab["surface_area"].div(combined_props_tab["volume"]))
rounded_scale = tuple(round(x, 2) for x in voxel_size_ZYX)
combined_props_tab.insert(1, "scale", str(rounded_scale))

display(combined_props_tab)

#### **4B. Skeleton metrics**

The next table summarizes `skan` skeleton metrics per PNN object and for the whole PNN in the image. No user input is required in this section.

In [ ]:
# ### PROCESSING - no edits below ###
# summarize skeleton branch table per skeleton object
paths_table_pruned.reset_index(inplace=True)
skel_sum1 = paths_table_pruned[['skeleton_id', 'branch_id']].groupby('skeleton_id').agg(['count'])
skel_sum2 = paths_table_pruned[['skeleton_id', 'branch_type']].groupby('skeleton_id').agg(['mean', 'median', 'min', 'max', 'std'])
skel_sum3 = paths_table_pruned[['skeleton_id', 'branch_distance', 'euclidean_distance']].groupby('skeleton_id').agg(['sum', 'mean', 'median', 'min', 'max', 'std'])

skel_summary = pd.concat([skel_sum1, skel_sum2, skel_sum3], axis=1)
skel_summary.columns = ['_'.join(col).strip() for col in skel_summary.columns.values]
skel_summary.reset_index(inplace=True)
skel_summary.insert(0, 'image_name', file_list[file_index].name)
skel_summary.insert(1, "scale", str(rounded_scale))
skel_summary.insert(2, 'object', 'PNN fragment')
skel_summary.rename(columns={'skeleton_id':'label',
                             'branch_id_count':'branch_count'}, inplace=True)

# summarize skeleton branch table for whole image
paths_table_pruned.insert(0, 'combined_skeleton_id', 1)  # assign all branches to skeleton ID 1 for whole image summary
combo_skel_sum1 = paths_table_pruned[['combined_skeleton_id', 'branch_id']].groupby('combined_skeleton_id').agg(['count'])
combo_skel_sum2 = paths_table_pruned[['combined_skeleton_id', 'branch_type']].groupby('combined_skeleton_id').agg(['mean', 'median', 'min', 'max', 'std'])
combo_skel_sum3 = paths_table_pruned[['combined_skeleton_id', 'branch_distance', 'euclidean_distance']].groupby('combined_skeleton_id').agg(['sum', 'mean', 'median', 'min', 'max', 'std'])

combo_skel_summary = pd.concat([combo_skel_sum1, combo_skel_sum2, combo_skel_sum3], axis=1)
combo_skel_summary.columns = ['_'.join(col).strip() for col in combo_skel_summary.columns.values]
combo_skel_summary.reset_index(inplace=True)
combo_skel_summary.insert(0, 'image_name', file_list[file_index].name)
combo_skel_summary.insert(1, "scale", str(rounded_scale))
combo_skel_summary.insert(2, 'object', 'whole PNN')
combo_skel_summary.rename(columns={'combined_skeleton_id':'label',
                                   'branch_id_count':'branch_count'}, inplace=True)

# combine both tables and format
final_skel_summary = pd.concat([skel_summary, combo_skel_summary], axis=0)

final_skel_summary

#### **4C. Combine tables above for output**

Run the next cell to merge object/intensity and skeleton summaries into one per-image output table.

In [ ]:
### PROCESSING - no edits below ###
combo = pd.merge(combined_props_tab, final_skel_summary, on= ['image_name', 'scale', 'object', 'label'], how='outer')

path_parts = file_list[file_index].rsplit("/")[-4:]
for i, label in enumerate(['experiment', 'sex', 'replicate', 'genotype']):
    combo.insert(i, label, path_parts[i])
combo.insert(4, 'file_path', file_list[file_index])

combo

----------

## **STEP 3. Batch Process segmentation and skeletonization (all images in one data folder):**

The above code is put together into a single function that enables you to batch process all of the images in a folder.

### **Segmentation and skeletonization**

In this step, segmentation and skeletonization are batch-processed for all images in one selected folder.

Specify the following information in the function call below:
- `file_path`: location of raw images to process
- `file_type`: input file extension (for example, `.tif`)
- `out_path`: output location for segmentation/skeleton files
- smoothing settings (`gaus_sigma`, `med_size`)
- one thresholding mode (manual, global auto, or local) with associated settings
- clean-up settings (`obj_min_diameter`, `obj_method`, `hole_min_diameter`, `hole_method`)
- `min_branch_len`: minimum endpoint branch length to keep

Then run the cell below to process the folder. Copy the cell to process additional folders sequentially.

In [ ]:
batch_PNN_seg_skel(file_path="/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/cKO",
                    file_type=".tif",
                    out_path="/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/20260309_test-final-funct_cKO-seg-skel",
                    gaus_sigma=2,
                    med_size=8,
                    manual_threshold_cutoff=0.045,
                    auto_threshold_method=None, #'multiotsu',
                    auto_threshold_adjust=None, #0.55,
                    auto_multiotsu_middle_to=None, #'background',
                    local_threshold_method=None, #'otsu',
                    local_threshold_adjust=None, #1,
                    local_threshold_size=None, #51, #71,
                    local_gaussian_sigma=None,
                    obj_min_diameter=10,
                    obj_method='3D',
                    hole_min_diameter=0,
                    hole_method='slices',
                    min_branch_len=0.3)

----------

## **STEP 4. Quality Check segmentations:**

### **Quality check output**

Before moving to quantification, inspect each segmentation/skeleton result and confirm it adequately represents the original intensity image.

Quality-check checklist:
1. Segmentation captures true PNN signal regions
2. Segmentation does not include excessive background
3. Skeleton follows the main morphology and is not dominated by short artifacts

If any files fail quality check, return to Step 2, update settings, and re-run Step 3 for that folder before proceeding.

----------

## **STEP 5. Batch Process quantification (all images in one data folder):**

### **Quantification**

Now that segmentation and skeletonization outputs have been quality checked, quantify PNN morphology features for all images in one folder.

Specify the following information in the function call below:
- `file_out_prefix`: prefix used to name the output quantification file
- `raw_file_path`: location of raw image files
- `raw_file_type`: raw image file extension (for example, `.tif`)
- `seg_skel_path`: location of matching segmentation and skeleton files
- `quant_out_path`: output location for quantification files
- `include_surface_area`: set to `True` to include surface area metrics (slower), or `False` to skip

Then run the cell below to process the folder. Copy the cell to process additional folders as needed.

In [ ]:
batch_PNN_quant(file_out_prefix="20260312",
                 raw_file_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/cKO", 
                 raw_file_type = ".tif",
                 seg_skel_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/20260225_test-auto-seg_cKO-seg-skel",
                 quant_out_path = "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/20260312_cKO-quant",
                 include_surface_area = True)

### **Batch Quantification Output Guide**

The `batch_PNN_quant` output CSV has one row per quantified object. The `object` column identifies whether the row summarizes:
- `PNN fragment`: one connected component from instance segmentation
- `whole PNN`: all segmented voxels merged into one object

The final table is produced by an **outer merge** between segmentation-derived and skeleton-derived summaries on:
- `image_name`, `scale`, `object`, `label`

Because this is an outer merge, some columns may be empty (`NaN`) when one side is missing for a given row.

#### **Metadata and grouping columns**
- `experiment`: experiment identifier parsed from `raw_file_path` (4th folder from end)
- `sex`: biological group parsed from `raw_file_path` (3rd folder from end)
- `replicate`: replicate identifier parsed from `raw_file_path` (2nd folder from end)
- `genotype`: condition/group parsed from `raw_file_path` (last folder)
- `file_path`: folder used as raw-image input for the run
- `image_name`: raw-image filename
- `scale`: voxel spacing tuple `(Z, Y, X)` stored as text
- `object`: row scope (`PNN fragment` or `whole PNN`)
- `label`: numeric object ID used to align segmentation and skeleton data

#### **Segmentation geometry columns (regionprops)**
- `bbox-0`, `bbox-1`, `bbox-2`: minimum bounding-box coordinates in `Z`, `Y`, `X`
- `bbox-3`, `bbox-4`, `bbox-5`: maximum bounding-box coordinates in `Z`, `Y`, `X` (exclusive upper bounds)
- `centroid-0`, `centroid-1`, `centroid-2`: object centroid coordinates in `Z`, `Y`, `X`
- `num_pixels`: number of voxels in the object (object size in voxel count)
- `volume`: physical object volume (`area` from regionprops, renamed)
- `equivalent_diameter`: diameter of a sphere with the same volume as the object
- `major_axis_length`: longest axis of the fitted ellipsoid-like shape
- `minor_axis_length`: shortest axis of the fitted ellipsoid-like shape
- `extent`: fraction of bounding-box volume occupied by object voxels (compact filling vs sparse occupancy)
- `solidity`: ratio of object volume to convex-hull volume (closer to 1 indicates fewer concavities)
- `euler_number`: topological descriptor of connectedness/holes (higher complexity can change this value)

#### **Intensity columns**
- `min_intensity`: lowest raw intensity among object voxels
- `max_intensity`: highest raw intensity among object voxels
- `mean_intensity`: average raw intensity among object voxels
- `intensity_std`: intensity variability within object voxels
- `intensity_sum`: integrated signal estimate (`mean_intensity * num_pixels`)

#### **Optional surface area columns** (`include_surface_area=True`)
- `surface_area`: surface area estimated from a marching-cubes mesh of the object
- `SA_to_volume_ratio`: `surface_area / volume`; higher values generally indicate more intricate, less compact object morphology

#### **Skeleton summary columns**
- `branch_count`: number of skeleton branches assigned to the object

- `branch_type_mean`, `branch_type_median`, `branch_type_min`, `branch_type_max`, `branch_type_std`:
  summary statistics of `skan` branch topology codes (see branch-type legend below)

- `branch_distance_sum`, `branch_distance_mean`, `branch_distance_median`, `branch_distance_min`, `branch_distance_max`, `branch_distance_std`:
  statistics for **geodesic branch length**, where geodesic length means the distance measured *along the skeleton path itself* (following all bends/turns).
  This reflects true path-traveled length through the neurite/PNN-like structure, not straight-line shortcut distance.

- `euclidean_distance_sum`, `euclidean_distance_mean`, `euclidean_distance_median`, `euclidean_distance_min`, `euclidean_distance_max`, `euclidean_distance_std`:
  statistics for straight-line endpoint-to-endpoint distance for each branch.

#### **Geodesic vs Euclidean distance (how to interpret)**
- Geodesic length: path length along the branch centerline
- Euclidean distance: straight-line distance between branch endpoints
- If geodesic is much larger than Euclidean, the branch is more tortuous/curved
- If geodesic is close to Euclidean, the branch is relatively straight

#### **Branch type code reference (`skan`)**
- `0`: endpoint-to-endpoint branch
- `1`: junction-to-endpoint branch
- `2`: junction-to-junction branch
- `3`: cycle/loop branch

----------

## **STEP 6. Summarize quantitative data per image (quantitative data from multiple folders):**

Once all biological replicates have undergone quantification, summarize the quantitative outputs per image (per cell for this dataset).

Specify the following information in the function call below:
- `out_file_prefix`: prefix for the summary output filename
- `csv_path_list`: list of quantification CSV file paths to include
- `out_path`: output location for the summary CSV

Then run the function-definition cell, then run the call cell to generate the summary table.

In [ ]:
batch_summarize_quant(out_file_prefix="20260312",
                       csv_path_list=["/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/20260312_cKO-quant/20260312-PNN_quantification.csv",
                                      "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 4/3D STED/20260312_WT-quant/20260312-PNN_quantification.csv"],
                                    #   "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 5/3D STED/20260312_cKO-quant/20260312-PNN_quantification.csv",
                                    #   "/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/Pair 5/3D STED/20260312_WT-quant/20260312-PNN_quantification.csv"],
                       out_path="/users/s/r/srhoads/PNN-morpho_Hayli-pilot-data/20260312_quant_summary")

### **Summary Output Guide**

This section explains how Step 6 transforms object-level quantification results (from Step 5) into grouped summary statistics that are ready for downstream analysis.

Each summary row corresponds to a unique combination of:
- `experiment`, `sex`, `replicate`, `genotype`, `image_name`, `scale`, `object`

This means summaries are computed separately for biological grouping variables, imaging scale, and object type.

#### **A. Grouping/identifier columns (carried into output rows)**
These columns define the identity of each summarized row:
- `experiment`: experiment-level dataset label
- `sex`: biological sex/group label
- `replicate`: biological replicate identifier
- `genotype`: genotype/condition label
- `image_name`: source image filename
- `scale`: voxel size tuple stored in the quantification table
- `object`: object class (`PNN fragment` or `whole PNN`)

Interpretation:
- Rows with different values in any of these fields are summarized independently.

#### **B. Object count and volume**
- `object_count`: number of objects contributing to that group
- `volume_sum`: total segmented volume across objects in the group
- `volume_mean`: average object volume
- `volume_median`: median object volume (more robust to outliers)
- `volume_std`: variability in object volume

Interpretation:
- Use `object_count` to understand sample size per grouped row.
- Compare `volume_sum` for total burden and `volume_mean/median` for typical object size.

#### **C. Morphology and intensity feature summaries (`_mean`, `_median`, `_std`)**
For each metric below, Step 6 generates three summary outputs:
- `<metric>_mean`
- `<metric>_median`
- `<metric>_std`

Metrics included:
- `num_pixels`: object voxel count
- `equivalent_diameter`: diameter of a sphere with equivalent volume
- `major_axis_length`, `minor_axis_length`: principal shape axes
- `extent`, `solidity`, `euler_number`: compactness/topology descriptors
- `min_intensity`, `max_intensity`, `mean_intensity`, `intensity_std`, `intensity_sum`: intensity distribution and integrated signal features

Interpretation:
- `_mean` and `_median` describe central tendency; `_std` captures heterogeneity within the grouped objects.

#### **D. Skeleton structure summaries (`_mean`, `_median`, `_std`)**
Skeleton-derived metrics are also summarized as mean/median/std for each grouped row:
- `branch_count`: number of branches per object
- `branch_type_mean`, `branch_type_median`, `branch_type_min`, `branch_type_max`, `branch_type_std`: branch topology patterns
- `branch_distance_sum`, `branch_distance_mean`, `branch_distance_median`, `branch_distance_min`, `branch_distance_max`, `branch_distance_std`: geodesic path-length features
- `euclidean_distance_sum`, `euclidean_distance_mean`, `euclidean_distance_median`, `euclidean_distance_min`, `euclidean_distance_max`, `euclidean_distance_std`: straight-line endpoint distance features

Interpretation:
- `branch_distance_*` reflects distance traveled along the branch path.
- `euclidean_distance_*` reflects straight-line endpoint separation.

#### **E. Optional surface-complexity summaries (included only when available in all inputs)**
If every input CSV includes surface metrics, Step 6 also outputs:
- `surface_area_mean`, `surface_area_median`, `surface_area_std`
- `SA_to_volume_ratio_mean`, `SA_to_volume_ratio_median`, `SA_to_volume_ratio_std`

Interpretation:
- `SA_to_volume_ratio` helps assess structural complexity/compactness in 3D.

#### **F. How to interpret geodesic vs Euclidean branch distances**
- **Geodesic branch length**: length measured along the branch centerline path (captures curvature/tortuosity)
- **Euclidean branch distance**: straight-line endpoint-to-endpoint distance

Practical interpretation:
- If geodesic values are much larger than Euclidean values, branches are more curved/tortuous.
- If geodesic and Euclidean values are similar, branches are relatively straight.

--------------------------
---------------------------
## **Versioning Notes:**
- **V1.1**: 12/11/2025 SR updates based on feedback from HSO
    - Load file (no change)
    - Segment PNN from WFA channel
        - Thresholding: update to add local and automated thresholding methods
    - Skeletonize PNN segmentation (no change)
    - Quantify PNN morphology & marker intensity:
        - Adding per PNN object and per PNN morphology quantification (skimage regionprops) and per object/PNN intensity quant of some additional channels (if they are the same resolution)

- **V1.0**: 10/28/2025 SR creates first draft with the following goals
    - Load file
        - list files of a specific type in path
        - read files (BioIO)
    - Segment PNN from WFA channel
        - Rescale intensities: min value = 0, max value = 1
        - Background subtraction: None
        - Denoising: None
        - Smoothing: gaussian = 2, median = 8
        - Thresholding: manual (low pass filter)
        - Clean-up: remove small objects (<10, 3D), fill small holes (none)
        - Instance segmentation: connectivity-based
    - Skeletonize PNN segmentation
        - Create skeleton object (skimage & skan)
        - Refine skeleton: remove small end-point branches (<1 um)
    - Quantify PNN morphology & marker intensity:
        - Intensity within entire PNN (segmented area, not per object)
        - Count and skeleton morphology metrics (from skan)